In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV


### Logistic Regression Model

In [ ]:
train_woe_trans_data = pd.read_parquet(
    "../data/processed/train_woe_output.parquet"
)

In [ ]:
train_df, test_df = train_test_split(
    train_woe_trans_data,
    test_size=0.2,
    random_state=0,
    shuffle=True
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
train_df.head(3)

In [ ]:
test_df.head(3)

In [ ]:
X_train = train_df.loc[:, ~train_df.columns.isin(["acct_id", "f_dpd_90plus_nxt_2yrs"])].copy()
y_train = train_df.loc[:, ["f_dpd_90plus_nxt_2yrs"]].copy()

X_test = test_df.loc[:, ~test_df.columns.isin(["acct_id", "f_dpd_90plus_nxt_2yrs"])].copy()
y_test = test_df.loc[:, ["f_dpd_90plus_nxt_2yrs"]].copy()

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
X_train.head(3)

In [ ]:
y_train.head(3)

In [ ]:
def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

In [ ]:
compute_vif(X_train)

In [ ]:
def backward_elimination(
    X: pd.DataFrame,
    y: pd.Series,
    significance_level: float = 0.05,
    vif_threshold: float = 5
):
    """
    Perform backward elimination with VIF check and Gini tracking.
    """
    X = sm.add_constant(X, has_constant="add")
    variables = list(X.columns)

    while True:
        # Fit logistic regression with current variables
        model = sm.Logit(y, X[variables]).fit(disp=0)
        y_pred = model.predict(X[variables])
        auc = roc_auc_score(y, y_pred)
        gini = 2 * auc - 1
        print(f"\nCurrent variables: {variables}")
        print(f"Gini: {gini:.4f}")

        # Compute VIF excluding constant
        vif = compute_vif(X[variables].drop(columns="const", errors="ignore"))
        print("VIF values:")
        print(vif)

        # Print p-values
        print("P-values:")
        print(model.pvalues)

        # Step 1: Drop variable with highest VIF if > threshold
        max_vif = vif["VIF"].max()
        if max_vif > vif_threshold:
            excluded_var = vif.loc[vif["VIF"].idxmax(), "feature"]
            variables.remove(excluded_var)
            print(f"Removing {excluded_var} due to high VIF ({max_vif:.2f})\n")
            continue

        # Step 2: Drop variable with highest p-value if > significance_level
        p_values = model.pvalues.drop("const", errors="ignore")
        max_pval = p_values.max()
        if max_pval > significance_level:
            excluded_var = p_values.idxmax()
            variables.remove(excluded_var)
            print(f"Removing {excluded_var} with p-value {max_pval:.4f}\n")
        else:
            break

    return model

In [ ]:
bkwelm_best_model = backward_elimination(X_train, y_train)

In [ ]:
bkwelm_best_model.summary()

#### Evaluation on Test Data

In [ ]:
X_test.head(3)

In [ ]:
# Predict probabilities on test set
X_test_const = sm.add_constant(X_test)
y_test_pred = bkwelm_best_model.predict(X_test_const)

auc = roc_auc_score(y_test, y_test_pred)
gini = 2 * auc - 1
print(100 * gini)

In [ ]:
y_test = pd.Series(y_test["f_dpd_90plus_nxt_2yrs"])
y_test = y_test.reset_index(drop=True)
y_test_pred = y_test_pred.reset_index(drop=True)

In [ ]:
def plot_cap(y_true, y_pred, title):
    total = len(y_true)
    class_1_count = np.sum(y_true)

    sorted_indices = np.argsort(y_pred)[::-1]
    y_true_sorted = y_true[sorted_indices]

    cum_pos = np.cumsum(y_true_sorted)

    x_values = np.arange(1, total+1) / total
    y_values = cum_pos / class_1_count

    plt.plot([0,1], [0,1], linestyle='--', color='grey', label="Random Model")
    plt.plot([0, class_1_count/total, 1], [0,1,1], linestyle='--', color='green', label="Perfect Model")

    # Your model CAP curve
    plt.plot(x_values, y_values, color='blue', label="Model")

    plt.title(title)
    plt.xlabel("Proportion of data (sorted by score)")
    plt.ylabel("Proportion of positives captured")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_cap(
    y_test,
    y_test_pred,
    title = "CAP Curve - Logistic Regression Model"
)

### Save Logit, Preprocessing & WOE Parameters to Disk

In [ ]:
model_path = "../models/logit_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(bkwelm_best_model, f)

print("Model saved successfully!")

In [ ]:
with open("../models/logit_model.pkl", "rb") as f:
    logit_model = pickle.load(f)

In [ ]:
print(logit_model.model.exog_names)

In [ ]:
preprocessing_params = {
    "imputation_values": {
        "monthly_income": 5400.0
    },

    "special_value_replacements": {
        "age": {
            0: 52.0
        },

        "n_dpd_90plus_hist": {
            96: 0.0,
            98: 0.0
        },

        "n_dpd_30_50_l2yrs": {
            96: 0.0,
            98: 0.0
        }
    }
}

# preprocessing_params = {
#     "imputation_values": {
#         "monthly_income": X_train["monthly_income"].median()
#     },

#     "special_value_replacements": {
#         "age": {
#             0: X_train.loc[X_train["age"] != 0, "age"].median()
#         },

#         "n_dpd_90plus_hist": {
#             96: 0.0,
#             98: 0.0
#         },

#         "n_dpd_30_50_l2yrs": {
#             96: 0.0,
#             98: 0.0
#         }
#     }
# }

### Testing 

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

In [3]:
with open("../models/preprocessing_params.pkl", "rb") as f:
    preprocessing_params = pickle.load(f)

with open("../models/logit_model.pkl", "rb") as f:
    logit_model = pickle.load(f)

In [4]:
print(logit_model.model.exog_names)

['const', 'n_dpd_90plus_hist_imp_woe', 'n_dpd_30_50_l2yrs_imp_woe', 'avg_util_unsec_woe', 'age_imp_woe', 'monthly_income_imp_woe', 'debt_income_ratio_woe']


In [6]:
new_customer = pd.DataFrame({
    "avg_util_unsec": [0.78],
    "n_dpd_90plus_hist": [0],
    "n_dpd_30_50_l2yrs": [98],
    "age": [61],
    "monthly_income": [5001.0],
    "debt_income_ratio": [0.81]
})

display(new_customer)

,avg_util_unsec,n_dpd_90plus_hist,n_dpd_30_50_l2yrs,age,monthly_income,debt_income_ratio
0,0.78,0,98,61,5001.0,0.81


In [45]:
def preprocess_data(input_df, preprocessing_params):

    output_df = input_df.copy()

    # Missing-value imputation
    output_df["monthly_income_imp"] = output_df["monthly_income"].fillna(
        preprocessing_params["imputation_values"]["monthly_income"]
    )

    # Special-value replacement
    output_df["age_imp"] = output_df["age"].replace(
        preprocessing_params["special_value_replacements"]["age"]
    )

    output_df["n_dpd_90plus_hist_imp"] = output_df["n_dpd_90plus_hist"].replace(
        preprocessing_params["special_value_replacements"]["n_dpd_90plus_hist"]
    )

    output_df["n_dpd_30_50_l2yrs_imp"] = output_df["n_dpd_30_50_l2yrs"].replace(
        preprocessing_params["special_value_replacements"]["n_dpd_30_50_l2yrs"]
    )

    return output_df

In [8]:
display(new_customer)
new_customer_preprocessed = preprocess_data(new_customer, preprocessing_params)
display(new_customer_preprocessed)

,avg_util_unsec,n_dpd_90plus_hist,n_dpd_30_50_l2yrs,age,monthly_income,debt_income_ratio
0,0.78,0,98,61,5001.0,0.81


,avg_util_unsec,n_dpd_90plus_hist,n_dpd_30_50_l2yrs,age,monthly_income,debt_income_ratio,monthly_income_imp,age_imp,n_dpd_90plus_hist_imp,n_dpd_30_50_l2yrs_imp
0,0.78,0,98,61,5001.0,0.81,5001.0,61,0,0


In [9]:
def woe_transformation_n_dpd_90plus_hist_imp(value):

    if value < 0.50:
        return 0.368445
    else:
        return -2.280864

def woe_transformation_n_dpd_30_50_l2yrs_imp(value):

    if value < 0.50:
        return 0.513985

    elif value < 1.50:
        return -0.903654

    else:
        return -1.865336

def woe_transformation_avg_util_unsec(value):

    if value < 0.06:
        return 1.395448

    elif value < 0.13:
        return 1.196037

    elif value < 0.22:
        return 0.801477

    elif value < 0.30:
        return 0.571255

    elif value < 0.39:
        return 0.264197

    elif value < 0.49:
        return 0.042306

    elif value < 0.70:
        return -0.380305

    elif value < 0.90:
        return -0.889439

    else:
        return -1.396248

def woe_transformation_age_imp(value):

    if value < 29.50:
        return -0.618478

    elif value < 36.50:
        return -0.502841

    elif value < 43.50:
        return -0.326797

    elif value < 47.50:
        return -0.210692

    elif value < 49.50:
        return -0.171605

    elif value < 52.50:
        return -0.130183

    elif value < 55.50:
        return -0.026589

    elif value < 59.50:
        return 0.253172

    elif value < 62.50:
        return 0.369651

    elif value < 67.50:
        return 0.726566

    elif value < 74.50:
        return 1.040110

    else:
        return 1.248390

def woe_transformation_monthly_income_imp(value):

    if value < 1508.50:
        return -0.099183

    elif value < 2569.50:
        return -0.394654

    elif value < 3331.50:
        return -0.469783

    elif value < 4833.50:
        return -0.229363

    elif value < 5333.50:
        return -0.062597

    elif value < 6643.50:
        return 0.117460

    elif value < 7656.50:
        return 0.197259

    elif value < 9945.50:
        return 0.281425

    else:
        return 0.460761

def woe_transformation_debt_income_ratio(value):

    if value < 0.02:
        return 0.297162

    elif value < 0.35:
        return 0.118882

    elif value < 0.42:
        return 0.057786

    elif value < 0.51:
        return -0.088438

    elif value < 0.65:
        return -0.321729

    elif value < 3.97:
        return -0.596973

    elif value < 995.50:
        return 0.063366

    else:
        return 0.328563


def apply_woe_transformation(input_df):

    output_df = input_df.copy()

    output_df["n_dpd_90plus_hist_imp_woe"] = output_df["n_dpd_90plus_hist_imp"].apply(woe_transformation_n_dpd_90plus_hist_imp)
    output_df["n_dpd_30_50_l2yrs_imp_woe"] = output_df["n_dpd_30_50_l2yrs_imp"].apply(woe_transformation_n_dpd_30_50_l2yrs_imp)
    output_df["avg_util_unsec_woe"] = output_df["avg_util_unsec"].apply(woe_transformation_avg_util_unsec)
    output_df["age_imp_woe"] = output_df["age_imp"].apply(woe_transformation_age_imp)
    output_df["monthly_income_imp_woe"] = output_df["monthly_income_imp"].apply(woe_transformation_monthly_income_imp)
    output_df["debt_income_ratio_woe"] = output_df["debt_income_ratio"].apply(woe_transformation_debt_income_ratio)

    return output_df

In [28]:
display(new_customer_preprocessed)
new_customer_preprocessed_woe = apply_woe_transformation(new_customer_preprocessed)
new_customer_preprocessed_woe

,avg_util_unsec,n_dpd_90plus_hist,n_dpd_30_50_l2yrs,age,monthly_income,debt_income_ratio,monthly_income_imp,age_imp,n_dpd_90plus_hist_imp,n_dpd_30_50_l2yrs_imp
0,0.78,0,98,61,5001.0,0.81,5001.0,61,0,0


,avg_util_unsec,n_dpd_90plus_hist,n_dpd_30_50_l2yrs,age,monthly_income,debt_income_ratio,monthly_income_imp,age_imp,n_dpd_90plus_hist_imp,n_dpd_30_50_l2yrs_imp,n_dpd_90plus_hist_imp_woe,n_dpd_30_50_l2yrs_imp_woe,avg_util_unsec_woe,age_imp_woe,monthly_income_imp_woe,debt_income_ratio_woe
0,0.78,0,98,61,5001.0,0.81,5001.0,61,0,0,0.368445,0.513985,-0.889439,0.369651,-0.062597,-0.596973


In [29]:
new_customer_preprocessed_woe.columns

Index(['avg_util_unsec', 'n_dpd_90plus_hist', 'n_dpd_30_50_l2yrs', 'age',
       'monthly_income', 'debt_income_ratio', 'monthly_income_imp', 'age_imp',
       'n_dpd_90plus_hist_imp', 'n_dpd_30_50_l2yrs_imp',
       'n_dpd_90plus_hist_imp_woe', 'n_dpd_30_50_l2yrs_imp_woe',
       'avg_util_unsec_woe', 'age_imp_woe', 'monthly_income_imp_woe',
       'debt_income_ratio_woe'],
      dtype='str')

In [30]:
print(logit_model.params)

const                       -2.636939
n_dpd_90plus_hist_imp_woe   -0.612542
n_dpd_30_50_l2yrs_imp_woe   -0.624596
avg_util_unsec_woe          -0.679693
age_imp_woe                 -0.472557
monthly_income_imp_woe      -0.214858
debt_income_ratio_woe       -0.665199
dtype: float64


In [31]:
model_variables = logit_model.params.index.tolist()
model_variables

['const',
 'n_dpd_90plus_hist_imp_woe',
 'n_dpd_30_50_l2yrs_imp_woe',
 'avg_util_unsec_woe',
 'age_imp_woe',
 'monthly_income_imp_woe',
 'debt_income_ratio_woe']

In [ ]:
new_customer_preprocessed_woe = sm.add_constant(
    new_customer_preprocessed_woe, 
    has_constant="add"
)
new_customer_preprocessed_woe.columns

Index(['const', 'avg_util_unsec', 'n_dpd_90plus_hist', 'n_dpd_30_50_l2yrs',
       'age', 'monthly_income', 'debt_income_ratio', 'monthly_income_imp',
       'age_imp', 'n_dpd_90plus_hist_imp', 'n_dpd_30_50_l2yrs_imp',
       'n_dpd_90plus_hist_imp_woe', 'n_dpd_30_50_l2yrs_imp_woe',
       'avg_util_unsec_woe', 'age_imp_woe', 'monthly_income_imp_woe',
       'debt_income_ratio_woe'],
      dtype='str')

In [42]:
new_customer_preprocessed_woe.columns

Index(['const', 'avg_util_unsec', 'n_dpd_90plus_hist', 'n_dpd_30_50_l2yrs',
       'age', 'monthly_income', 'debt_income_ratio', 'monthly_income_imp',
       'age_imp', 'n_dpd_90plus_hist_imp', 'n_dpd_30_50_l2yrs_imp',
       'n_dpd_90plus_hist_imp_woe', 'n_dpd_30_50_l2yrs_imp_woe',
       'avg_util_unsec_woe', 'age_imp_woe', 'monthly_income_imp_woe',
       'debt_income_ratio_woe'],
      dtype='str')

In [43]:
X = new_customer_preprocessed_woe[model_variables]
X.columns

Index(['const', 'n_dpd_90plus_hist_imp_woe', 'n_dpd_30_50_l2yrs_imp_woe',
       'avg_util_unsec_woe', 'age_imp_woe', 'monthly_income_imp_woe',
       'debt_income_ratio_woe'],
      dtype='str')

In [44]:
logit_model.predict(X)

0    0.087605
dtype: float64

In [48]:
new_customer = pd.DataFrame({
    "avg_util_unsec": [0.78],
    "n_dpd_90plus_hist": [0],
    "n_dpd_30_50_l2yrs": [98],
    "age": [61],
    "monthly_income": [5001.0],
    "debt_income_ratio": [0.81]
})

display(new_customer)

,avg_util_unsec,n_dpd_90plus_hist,n_dpd_30_50_l2yrs,age,monthly_income,debt_income_ratio
0,0.78,0,98,61,5001.0,0.81


In [54]:
def predict_pd(input_df, preprocessing_params, logit_model):
    """
    Preprocess the input data, apply WOE transformation, and predict default probability using the logistic regression model.
    """
    preprocessed_df = preprocess_data(input_df, preprocessing_params)
    woe_transformed_df = apply_woe_transformation(preprocessed_df)

    woe_transformed_df = sm.add_constant(
        woe_transformed_df, 
        has_constant="add"
    )

    model_variables = logit_model.params.index.tolist()
    X = woe_transformed_df[model_variables]
    predicted_probability = logit_model.predict(X)

    return predicted_probability

In [55]:
predict_pd(new_customer, preprocessing_params, logit_model)

0    0.087605
dtype: float64